In [0]:

# ###################################  DELETAR ###########################################
# # ─────────────────────────────────────────────────────────────────────
# # LÊ A CONTROL TABLE (a "lista de tarefas" — quais tabelas carregar)
# # ─────────────────────────────────────────────────────────────────────

# # spark.table(...) lê uma tabela do Unity Catalog como DataFrame.
# # .filter("is_active = true") pega só as tabelas ligadas (ignora as desativadas).
# # .collect() traz as linhas do Spark (distribuído) pra memória do driver,
# #   virando uma lista de Rows do Python — dá pra fazer um for normal em cima.
# #   (Ok aqui porque são só 14 linhas de config, dado minúsculo.)
# ctrl = spark.table("hpn.metadata.control_ingestion").filter("is_active = true").collect()

# # Caminho base do container landing no ADLS. Guardo numa variável pra não
# # repetir a URL gigante em cada iteração (fica mais limpo e evita erro de digitação).
# LANDING = "abfss://landing@adlshpn.dfs.core.windows.net"


# # ─────────────────────────────────────────────────────────────────────
# # LOOP: uma volta por tabela da control table
# # ─────────────────────────────────────────────────────────────────────

# # 'row' é cada linha da control table (um dicionário-like com as colunas).
# for row in ctrl:

#     # row["coluna"] pega o valor daquela coluna NAQUELA linha.
#     tbl    = row["source_table"]     # nome da tabela (ex. "customer")
#     folder = row["target_folder"]    # pasta no landing (ex. "customer")
#     mode   = row["ingestion_mode"]   # "copy_into" ou "autoloader"

#     # f"..." é f-string: injeta o valor das variáveis dentro do texto.
#     src    = f"{LANDING}/{folder}"          # origem: pasta no landing
                                            
#     target = f"hpn.1_bronze.{tbl}"        # destino: tabela na bronze


#     # ─── CAMINHO 1: COPY INTO (o padrão, batch idempotente) ───
#     if mode == "copy_into":

#         # Cria a tabela VAZIA se ela ainda não existir.
#         # O COPY INTO precisa de uma tabela-destino existente pra escrever.
#         # Sem colunas por enquanto — elas vêm dos Parquet no próximo comando.
#         spark.sql(f"CREATE TABLE IF NOT EXISTS {target}")

#         # O COPY INTO em si:
#         #   FROM '{src}'                → lê os Parquet da pasta no landing
#         #   FILEFORMAT = PARQUET        → diz o formato dos arquivos
#         #   mergeSchema = true          → deixa o COPY INTO INFERIR as colunas
#         #                                 do Parquet e criá-las na tabela vazia
#         # Idempotência automática: ele guarda quais arquivos já carregou;
#         #   se rodar de novo, pula os repetidos (não duplica).
#         spark.sql(f"""
#             COPY INTO {target}
#             FROM '{src}'
#             FILEFORMAT = PARQUET
#             COPY_OPTIONS ('mergeSchema' = 'true')
#         """)

#         # Conta as linhas da tabela recém-carregada só pra logar/validar.
#         cnt = spark.table(target).count()

#         # Imprime um resumo. {tbl:<22} alinha o nome à esquerda em 22 espaços
#         #   (só pra ficar bonitinho/alinhado no output).
#         print(f"[copy_into]  {tbl:<22} OK  ({cnt} linhas)")

#     # ─── CAMINHO 2: AUTO LOADER (opt-in, pra exceções de alto volume/streaming) ───
#     elif mode == "autoloader":

#         # Auto Loader precisa de 2 locais de controle (guardados num Volume do UC):
#         chk = f"/Volumes/hpn/1_bronze/ops/{tbl}_chk"      # checkpoint: o que já processou
#         sch = f"/Volumes/hpn/1_bronze/ops/{tbl}_schema"   # schema inferido

#         # Monta a LEITURA em streaming (lazy — ainda não lê nada):
#         df = (spark.readStream                             # modo streaming/incremental
#               .format("cloudFiles")                        # ativa o Auto Loader
#               .option("cloudFiles.format","parquet")       # arquivos são Parquet
#               .option("cloudFiles.schemaLocation", sch)    # onde guardar o schema
#               .load(src))                                  # pasta de origem

#         # Monta a ESCRITA e DISPARA (aqui sim executa):
#         (df.writeStream
#            .option("checkpointLocation", chk)             # onde salvar o progresso
#            .trigger(availableNow=True)                     # processa o que tem e PARA
#            .toTable(target)                                # grava na tabela bronze
#          ).awaitTermination()                              # espera terminar antes de
#                                                            #   seguir pra próxima tabela
#         print(f"[autoloader] {tbl:<22} OK")

#     # ─── CAMINHO 3: valor inesperado em ingestion_mode ───
#     else:
#         # Se alguém colocar um modo que não existe, não quebra o loop —
#         # só avisa e pula pra próxima tabela.
#         print(f"[skip]       {tbl:<22} modo desconhecido: {mode}")


In [0]:
%sql
SELECT COUNT(*) total, COUNT(DISTINCT id) distintos   -- ajuste a PK real
FROM hpn.1_bronze.customer;


In [0]:
files = dbutils.fs.ls("abfss://landing@adlshpn.dfs.core.windows.net/customer")
for f in files:
    print(f.name, f.size)


In [0]:
tbl    = "customer"
src    = f"abfss://landing@adlshpn.dfs.core.windows.net/{tbl}"
target = f"hpn.`1_bronze`.{tbl}"

# 1) DIAGNÓSTICO: quantos .parquet tem no landing/customer?
files = [f.name for f in dbutils.fs.ls(src) if f.name.endswith(".parquet")]
print("Arquivos no landing:", files, "\n")

# 2) DROP: zera a tabela E o histórico de COPY INTO
#    (o "livro-caixa" do COPY INTO vive nos metadados da tabela; dropar limpa tudo)
spark.sql(f"DROP TABLE IF EXISTS {target}")

# 3) FULL LOAD via OVERWRITE (o design correto pra load_type=full)
#    read = lê tudo que está no landing; overwrite = SUBSTITUI a bronze (nunca acumula)
df = spark.read.format("parquet").load(src)
(df.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(target))

# 4) VALIDA
tot = spark.table(target).count()
print(f"{tbl}: {tot} linhas")


In [0]:
# ─────────────────────────────────────────────────────────────────────
# LÊ A CONTROL TABLE (a "lista de tarefas" — quais tabelas carregar)
# ─────────────────────────────────────────────────────────────────────

# spark.table(...) lê uma tabela do Unity Catalog como DataFrame.
# .filter("is_active = true") pega só as tabelas ligadas (ignora as desativadas).
# .collect() traz as linhas do Spark (distribuído) pra memória do driver,
#   virando uma lista de Rows do Python — dá pra fazer um for normal em cima.
#   (Ok aqui porque são só 14 linhas de config, dado minúsculo.)

ctrl = spark.table("hpn.metadata.control_ingestion").filter("is_active = true").collect()

# Caminho base do container landing no ADLS. Guardo numa variável pra não
# repetir a URL gigante em cada iteração (fica mais limpo e evita erro de digitação).
LANDING = "abfss://landing@adlshpn.dfs.core.windows.net"

# 'row' é cada linha da control table (um dicionário-like com as colunas).
for row in ctrl:
    tbl       = row["source_table"]               # nome da tabela (ex. "customer")
    folder    = row["target_folder"]              # pasta no landing (ex. "customer")
    load_type = row["load_type"]                  # full ou incremental 
    mode      = row["ingestion_mode"]             ## "copy_into" ou "autoloader"
    src       = f"{LANDING}/{folder}"             #   # f"..." é f-string: injeta o valor das variáveis dentro do texto.
    target    = f"hpn.`1_bronze`.{tbl}"           # destino: tabela na bronze

    # ── FULL: substitui a foto inteira (idempotente, nunca duplica) ──
    if load_type == "full":
        df = spark.read.format("parquet").load(src)
        (df.write.format("delta").mode("overwrite")
           .option("overwriteSchema","true").saveAsTable(target))
        print(f"[full]  {tbl:<22} OK ({spark.table(target).count()} linhas)")

    # ── INCREMENTAL: acrescenta só o novo ──
    elif load_type == "incremental":
        # ─── CAMINHO 1: COPY INTO (o padrão, batch idempotente) ───
        if mode == "copy_into":
            # Cria a tabela VAZIA se ela ainda não existir.
            # O COPY INTO precisa de uma tabela-destino existente pra escrever.
            # Sem colunas por enquanto — elas vêm dos Parquet no próximo comando.
            spark.sql(f"CREATE TABLE IF NOT EXISTS {target}")

        # O COPY INTO em si:
        #   FROM '{src}'                → lê os Parquet da pasta no landing
        #   FILEFORMAT = PARQUET        → diz o formato dos arquivos
        #   mergeSchema = true          → deixa o COPY INTO INFERIR as colunas
        #                                 do Parquet e criá-las na tabela vazia
        # Idempotência automática: ele guarda quais arquivos já carregou;
        #   se rodar de novo, pula os repetidos (não duplica).
            spark.sql(f"""COPY INTO {target} FROM '{src}'
                          FILEFORMAT = PARQUET
                          COPY_OPTIONS ('mergeSchema'='true')""")
            print(f"[incr/copy_into]  {tbl:<22} OK ({spark.table(target).count()} linhas)")

    # ─── CAMINHO 2: AUTO LOADER (opt-in, pra exceções de alto volume/streaming) ───
        elif mode == "autoloader":
            chk = f"/Volumes/hpn/1_bronze/ops/{tbl}_chk"      # checkpoint: o que já processou
            sch = f"/Volumes/hpn/1_bronze/ops/{tbl}_schema"   # schema inferido

            # Monta a LEITURA em streaming (lazy — ainda não lê nada):
            df = (spark.readStream
                  .format("cloudFiles")                                   # ativa o Auto Loader
                  .option("cloudFiles.format","parquet")                  # arquivos são Parquet
                  .option("cloudFiles.schemaLocation", sch)               # onde guardar o schema
                  .load(src))                                             # pasta de origem
            
            # Monta a ESCRITA e DISPARA (aqui sim executa):
            (df.writeStream
               .option("checkpointLocation", chk)                                 # onde salvar o progresso
               .trigger(availableNow=True)                                        # processa o que tem e PARA
               .toTable(target)                                                   # grava na tabela bronze
             ).awaitTermination()                                                  # espera terminar antes de continuar
            print(f"[incr/autoloader] {tbl:<22} OK")
    else:
         # ─── CAMINHO 3: valor inesperado em ingestion_mode ───
        # Se alguém colocar um modo que não existe, não quebra o loop —
        # só avisa e pula pra próxima tabela.
        print(f"[skip]  {tbl:<22} load_type desconhecido: {load_type}")


In [0]:
%sql
select * from hpn.1_bronze.customer limit 10